# E-commerce Business Dashboard Analysis  
**BigQuery + Looker Studio + Python (Colab)**

This project analyzes an e-commerce business using public data from `bigquery-public-data.thelook_ecommerce`.

The goal is to build a business-facing analytics dashboard and validate the key metrics and trends in Python. The analysis focuses on four main areas:

- Business performance overview
- Customer retention
- Product performance
- Customer behavior and conversion

The project combines:
- **SQL in BigQuery** for data extraction and metric calculation
- **Looker Studio** for dashboarding
- **Python / Plotly in Colab** for validation, storytelling, and custom visualizations

## Data Source

This analysis uses the public BigQuery dataset:

`bigquery-public-data.thelook_ecommerce`

The main tables used in this project are:

- `orders` — order-level information such as order date, customer, shipment, and status
- `order_items` — item-level revenue data
- `products` — product and category information
- `inventory_items` — product cost data used for profit and margin calculations
- `events` — user behavior and funnel events

## Dataset Notes

The `thelook_ecommerce` dataset is a synthetic dataset designed for demonstration and analytical practice.  
As a result, some patterns in the data may appear unrealistic compared to a real e-commerce business. For example:

- extremely large product assortments with very low sales per individual SKU,
- inconsistent product naming or category assignments,
- certain metrics or distributions that do not fully reflect real business dynamics.

These limitations should be considered when interpreting the results.

The purpose of this project is not to draw real business conclusions from this dataset, but rather to demonstrate practical skills in:

- analytical data modeling,
- SQL-based metric calculation,
- dashboard design,
- and data visualization in Python.

## Analytical Scope

The notebook is designed as the analytical companion to the final dashboard.

While the Looker Studio dashboard presents the final business view, this notebook documents:
- how the data was extracted,
- and how the visual insights were built in Python.

## Environment Setup

This section initializes the Python environment and connects to BigQuery.

In [1]:
#@title Insert your project_id
project_id = 'your-project-id-here' # @param {type:"string"}

from google.colab import auth
from google.cloud import bigquery

# Authorization
auth.authenticate_user()

# Create a BigQuery client

client = bigquery.Client(project=project_id)


import pandas as pd
import datetime
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

# 1. Business Overview

This section summarizes the overall business performance using high-level KPIs and trend analysis.

The objective is to answer the following questions:

- Is the business growing?
- How do current results compare with the previous period?
- Is growth driven by order volume, customer growth, or average order value?
- How profitable is the current business performance?

## 1.1 Data Extraction

The following query prepares the KPI dataset used for the executive overview, including revenue, profit, margin, average order value, customer metrics, and period-over-period comparison.

In [4]:
query_orders_analytics = """
WITH order_headers AS (
  SELECT
    order_id,
    user_id,
    DATE(created_at) AS created_at,
    DATE(shipped_at) AS shipped_at,
    DATE(delivered_at) AS delivered_at,
    num_of_item
  FROM `bigquery-public-data.thelook_ecommerce.orders`
  WHERE status NOT IN ('Cancelled', 'Returned')
),

order_details AS (
  SELECT
    oi.order_id,
    oi.user_id,
    oi.product_id,
    oi.sale_price,
    i.cost
    FROM `bigquery-public-data.thelook_ecommerce.order_items` oi
    LEFT JOIN `bigquery-public-data.thelook_ecommerce.inventory_items` i
    ON oi.product_id = i.product_id
    AND i.id = oi.inventory_item_id
    WHERE status NOT IN ('Cancelled', 'Returned')
),

order_totals AS (
  SELECT
    order_id,
    user_id,
    SUM(sale_price) AS order_amount,
    SUM(IFNULL(cost, 0)) AS order_cost
  FROM order_details
  GROUP BY 1, 2
),

orders_enriched AS (
  SELECT
    h.order_id,
    h.user_id,
    h.created_at,
    h.shipped_at,
    h.delivered_at,
    h.num_of_item,
    r.order_cost,
    r.order_amount,
    ROW_NUMBER() OVER (PARTITION BY h.user_id ORDER BY h.created_at) AS order_number
  FROM order_headers h
  JOIN order_totals r
    USING (order_id, user_id)
)

SELECT *
FROM orders_enriched

"""

df = client.query(query_orders_analytics).to_dataframe()

In [5]:
# Convert dates if they are not already in datetime format
df['created_at'] = pd.to_datetime(df['created_at'])

# Define the boundaries of the periods
today = df['created_at'].max()
last_30_days_start = today - datetime.timedelta(days=29) # including today
prev_30_days_start = last_30_days_start - datetime.timedelta(days=30) # not including the last day of last period

# Filter two periods
df_current = df[df['created_at'] >= last_30_days_start]
df_prev = df[(df['created_at'] >= prev_30_days_start) & (df['created_at'] < last_30_days_start)]

def get_metrics(data):
    rev = data['order_amount'].sum()
    cost = data['order_cost'].sum()
    profit = rev - cost
    orders = data['order_id'].nunique()
    items = data['num_of_item'].sum()
    customers = data['user_id'].nunique()
    new_cust = data[data['order_number'] == 1]['user_id'].nunique()

    return {
        'revenue': rev,
        'profit': profit,
        'margin': (profit / rev * 100) if rev > 0 else 0,
        'aov': rev / orders if orders > 0 else 0,
        'ppo': profit / orders if orders > 0 else 0,
        'orders': orders,
        'customers': customers,
        'new_customers': new_cust,
        'repeat_rate': ((customers - new_cust) / customers * 100) if customers > 0 else 0,
        'items': items
    }

curr = get_metrics(df_current)
prev = get_metrics(df_prev)

## 1.2 Executive KPI Summary

The KPI cards below compare the current 30-day period with the previous 30-day period and highlight the most important business indicators:

- Revenue
- Gross Profit
- Gross Margin
- Average Order Value
- Profit per Order
- Orders
- Customers
- New Customers
- Repeat Customer Rate
- Items Sold

In [6]:
#@title KPI Summary

# List of metrics for easy iteration (Title, Current, Previous, Format)
metrics_map = [
    # Cтрока 1
    ("Revenue", curr['revenue'], prev['revenue'], "$"),
    ("Gross Profit", curr['profit'], prev['profit'], "$"),
    ("Gross Margin", curr['margin'], prev['margin'], "%"),
    ("Avg Order Value", curr['aov'], prev['aov'], "$"),
    ("Profit per Order", curr['ppo'], prev['ppo'], "$"),
    # Строка 2
    ("Orders", curr['orders'], prev['orders'], ""),
    ("Customers", curr['customers'], prev['customers'], ""),
    ("New Customers", curr['new_customers'], prev['new_customers'], ""),
    ("Repeat Cust. Rate", curr['repeat_rate'], prev['repeat_rate'], "%"),
    ("Items Sold", curr['items'], prev['items'], "")
]

fig = make_subplots(
    rows=2, cols=5,
    specs=[[{'type': 'indicator'}] * 5] * 2,
    vertical_spacing=0.15
)

for i, (name, val, p_val, unit) in enumerate(metrics_map):
    row = 1 if i < 5 else 2
    col = (i % 5) + 1

   # Prefix/suffix calculation
    val_format = unit + ":.2s" if unit == "$" else ".2s"
    if unit == "%": val_format = ".2f"

    fig.add_trace(go.Indicator(
        mode = "number+delta",
        value = val,
        title = {"text": name, "font": {"size": 16}},
        delta = {'reference': p_val, 'relative': True, 'valueformat': ".1%"},
        number = {'prefix': unit if unit == "$" else "",
                  'suffix': "%" if unit == "%" else "",
                  'valueformat': ".3s" if val > 1000 else ".2f"},
    ), row=row, col=col)

fig.update_layout(
    height=400,
    margin=dict(t=50, b=50, l=20, r=20),
    template="plotly_white"
)

fig.show()

## 1.3 Revenue and Order Trends

To complement the KPI summary, the following charts compare the current period with the same period in the previous year.

These visualizations help identify:
- seasonality,
- growth patterns,
- and differences between revenue growth and order growth.

*Note: The dataset used in this project is synthetic (`thelook_ecommerce`), so some trends may appear less realistic than in a real e-commerce business.*

In [11]:
#@title Revenue and Order Trends


# Defining boundaries of the periods
current_start = pd.to_datetime('2025-01-01')
current_end = df['created_at'].max()
last_year_start = current_start - pd.DateOffset(years=1)
last_year_end = current_end - pd.DateOffset(years=1)

# Filtering data
df_this_year = df[(df['created_at'] >= current_start) & (df['created_at'] <= current_end)].copy()
df_last_year = df[(df['created_at'] >= last_year_start) & (df['created_at'] <= last_year_end)].copy()

# Group by days
daily_this = df_this_year.groupby(df_this_year['created_at'].dt.date).agg({'order_amount': 'sum', 'order_id': 'nunique'}).reset_index()
daily_last = df_last_year.groupby(df_last_year['created_at'].dt.date).agg({'order_amount': 'sum', 'order_id': 'nunique'}).reset_index()

# Bring the dates of last year to 2025 for overlapping
daily_last['display_date'] = pd.to_datetime(daily_last['created_at']) + pd.DateOffset(years=1)
daily_this['display_date'] = pd.to_datetime(daily_this['created_at'])

# Create subplots (2 rows, 1 column)
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Revenue Trend: Current vs Last Year", "Orders Trend: Current vs Last Year"),
                    vertical_spacing=0.15)

# --- CHART 1: REVENUE ---
# Last Year (Pale Line)
fig.add_trace(go.Scatter(x=daily_last['display_date'],
                         y=daily_last['order_amount'],
                         name='Revenue (Last Year)',
                         line=dict(color='lightgrey', width=2),
                         hovertemplate="Last Year: $%{y:,.0f}<extra></extra>"
                         ), row=1, col=1)
# Current Year (Bright Line)
fig.add_trace(go.Scatter(x=daily_this['display_date'],
                         y=daily_this['order_amount'],
                         name='Revenue (Current Year)',
                         line=dict(color='#1f77b4', width=3),
                         hovertemplate="Current Year: $%{y:,.0f}<extra></extra>"
                         ), row=1, col=1)


# --- CHART 2: ORDERS ---
# Last year (Pale line)
fig.add_trace(go.Scatter(x=daily_last['display_date'],
                         y=daily_last['order_id'],
                         name='Orders (Last Year)',
                         line=dict(color='lightgrey', width=2),
                         hovertemplate="Last Year: %{y}<extra></extra>"
                         ), row=2, col=1)
# Current Year (Bright Line)
fig.add_trace(go.Scatter(x=daily_this['display_date'],
                         y=daily_this['order_id'],
                         name='Orders (Current Year)',
                         line=dict(color='#ff7f0e', width=3),
                         hovertemplate="Current Year: %{y}<extra></extra>"
                         ), row=2, col=1)

# Design settings
fig.update_layout(
    height=700,
    template="plotly_white",
    showlegend=True,
    title_text="Year-over-Year Performance Analysis",
    hovermode="x unified",
    hoverlabel=dict(bgcolor="white", font_size=12)
    )

# Improve the X-axis format (to display months)
fig.update_xaxes(tickformat="%b %d")

fig.show()

### Key Takeaways

- Revenue growth should be interpreted together with order growth and AOV.
- If revenue grows faster than orders, average order value may be improving.
- If orders grow but profit margin declines, the business may be scaling less efficiently.

- Customer growth provides important context for revenue trends, as revenue expansion can be driven either by acquiring new customers or by increasing activity among existing ones.
- A rising share of new customers may indicate strong acquisition performance, but it should be monitored together with retention metrics to assess long-term customer value.
- Repeat customer rate helps evaluate customer loyalty and the sustainability of growth beyond initial purchases.

# 2. Customer Retention

This section focuses on repeat purchase behavior and customer retention over time.

The main questions are:

- How often do customers return after their first purchase?
- Are more recent customer cohorts performing better or worse?
- Is customer acquisition translating into retention?

## 2.1 Data Extraction

The following query builds monthly customer cohorts based on first purchase date and calculates retention by month index.

In [12]:
query_cohort_retention = """
WITH user_first_order AS (
  SELECT
    user_id,
    DATE_TRUNC(DATE(MIN(created_at)), MONTH) AS first_order_month
  FROM `bigquery-public-data.thelook_ecommerce.orders`
  WHERE DATE(created_at) BETWEEN DATE '2025-01-01' AND DATE '2025-12-31'
    AND status NOT IN ('Cancelled', 'Returned')
  GROUP BY user_id
),

user_orders AS (
  SELECT DISTINCT
    user_id,
    DATE_TRUNC(DATE(created_at), MONTH) AS order_month
  FROM `bigquery-public-data.thelook_ecommerce.orders`
  WHERE status NOT IN ('Cancelled', 'Returned')
    AND DATE(created_at) >= DATE '2025-01-01'
),

cohort_data AS (
  SELECT
    u.user_id,
    f.first_order_month,
    u.order_month,
    DATE_DIFF(u.order_month, f.first_order_month, MONTH) AS month_index
  FROM user_orders u
  JOIN user_first_order f
    USING (user_id)
),

cohort_size AS (
  SELECT
    first_order_month,
    COUNT(DISTINCT user_id) AS cohort_size
  FROM cohort_data
  WHERE month_index = 0
  GROUP BY 1
),

retention AS (
  SELECT
    first_order_month,
    month_index,
    COUNT(DISTINCT user_id) AS users
  FROM cohort_data
  GROUP BY 1, 2
)

SELECT
  r.first_order_month,
  r.month_index,
  r.users AS active_users,
  c.cohort_size,
  ROUND(SAFE_DIVIDE(r.users, c.cohort_size), 4) AS retention_rate,
  ROUND(SAFE_DIVIDE(r.users, c.cohort_size) * 100, 1) AS retention_rate_pct
FROM retention r
JOIN cohort_size c
  USING (first_order_month)
WHERE r.month_index BETWEEN 0 AND 18
ORDER BY r.first_order_month, r.month_index;


"""
df_cohort = client.query(query_cohort_retention).to_dataframe()

## 2.2 Cohort Retention Analysis

The heatmap below shows customer retention by monthly cohort.

Each row represents customers acquired in a given month, and each column shows the share of that cohort returning in subsequent months.

This visualization helps identify:
- retention decay,
- stronger or weaker cohorts,
- and longer-term repeat purchase behavior.

## Average Retention Curve

In addition to cohort-level retention, the chart below shows the average retention trend across cohorts.

This provides a simplified view of how quickly retention declines after the first purchase and whether it stabilizes over time.

In [9]:
#@title Cohort Retention Analysis

# Data Preparation (Pivot Table)
retention_pivot = df_cohort.pivot(
    index='first_order_month',
    columns='month_index',
    values='retention_rate_pct'
).fillna(0)


# Data Preparation
retention_pivot = df_cohort.pivot(index='first_order_month', columns='month_index', values='retention_rate_pct')
avg_retention = df_cohort.groupby('month_index')['retention_rate_pct'].mean().to_frame().T
avg_retention.index = ['<b>AVERAGE</b>']
avg_retention = avg_retention.fillna(0)

# Create subplots
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.02, row_heights=[0.85, 0.15])


# Setting up custom text for the pop-up window
# %{y} is the Y-axis value (date), %{x} is the X-axis value (index), %{z} is the number itself
hover_text = (
    "<b>First Purchase:</b> %{y}<br>" +
    "<b>Months Since:</b> %{x}<br>" +
    "<b>Retention:</b> %{z:.1f}%" +
    "<extra></extra>" # Removes the standard sidebar (trace name)
)

# Contrast scale adjustment:
# Set zmax=15 so that everything above 15% is as dark as possible,
# and the difference between 2% and 7% becomes visually huge.
z_max_limit = 15

# --- Heatmap ---
fig.add_trace(go.Heatmap(
    z=retention_pivot.values,
    x=retention_pivot.columns,
    y=retention_pivot.index.astype(str),
    colorscale='Blues',
    zmin=0, zmax=z_max_limit,
    text=retention_pivot.values,
    texttemplate="%{text:.1f}%",
    showscale=False,
    hovertemplate=hover_text, # Apply our template
    colorbar=dict(title="Retention %", tickvals=[0, 1, 2, 3, 4, 5, 7, 10, 15])
), row=1, col=1)

# --- Average Row ---
fig.add_trace(go.Heatmap(
    z=avg_retention.values,
    x=avg_retention.columns,
    y=avg_retention.index,
    colorscale='Blues',
    zmin=0, zmax=z_max_limit,
    text=avg_retention.values,
    texttemplate="%{text:.1f}%",
    showscale=False,
    hovertemplate="<b>%{y}</b><br>Month: %{x}<br>Avg: %{z:.1f}%<extra></extra>",
), row=2, col=1)

# 3. Оформление
fig.update_layout(
    title='Cohort Retention Analysis',
    width=1100, height=700, template='plotly_white'
)
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_xaxes(title_text="Months Since First Purchase", row=2, col=1, dtick=1)
fig.show()

## 2.3 New Customer Acquisition Trend

Retention should be interpreted together with acquisition volume.

The following chart shows how many new customers were acquired over time, helping distinguish between growth driven by acquisition and growth supported by repeat behavior.

In [19]:
#@title New Customer Acquisition Trend

# Calculate the total number of unique users by month
total_users = df.groupby(df['created_at'].dt.to_period('M'))['user_id'].nunique()

# Only count new customers (those for whom this is their first order)
new_users = df[df['order_number'] == 1].groupby(df['created_at'].dt.to_period('M'))['user_id'].nunique()

# Collect into the final DataFrame
df_new_users_monthly = pd.DataFrame({
    'Month': total_users.index.astype(str),
    'New Users': new_users.values,
    'Total Active': total_users.values
})

# Calculate the percentage
df_new_users_monthly['New Users %'] = (df_new_users_monthly['New Users'] / df_new_users_monthly['Total Active'] * 100).round(2)


fig = go.Figure()

# Bars for quantity
fig.add_trace(go.Bar(
    x=df_new_users_monthly['Month'],
    y=df_new_users_monthly['New Users'],
    name='New Users Count',
    marker_color='royalblue',
    hovertemplate="New Users Count: %{y:,.0f}<extra></extra>"
))

# Line for percentage (on separate Y-axis)
fig.add_trace(go.Scatter(
    x=df_new_users_monthly['Month'],
    y=df_new_users_monthly['New Users %'],
    name='New Users %',
    yaxis='y2',
    line=dict(color='orange', width=3),
    hovertemplate="New Users %: %{y:.1f}%<extra></extra>"
))

fig.update_layout(
    title='New Users Acquisition Trend',
    yaxis=dict(title='Number of New Users'),
    yaxis2=dict(title='Percentage of Total Users', overlaying='y', side='right', range=[0, 100]),
    template='plotly_white',
    hovermode="x unified",
    hoverlabel=dict(bgcolor="white", font_size=12),
    barcornerradius=15
)

fig.show()

### Key Takeaways

- Monthly repeat purchase rates are relatively low, which is typical for e-commerce.
- Some cohorts appear to retain slightly better than others, suggesting variation in customer quality or seasonality.
- Retention and acquisition should be analyzed together to understand sustainable growth.
- The number of newly acquired users shows a clear upward trend, indicating strong growth in customer acquisition.
- At the same time, the share of new users gradually declines, suggesting that the base of returning customers is expanding as the business matures.

# 3. Product Performance

This section evaluates which categories and products contribute most to revenue and profitability.

The objective is to answer:

- Which categories drive the business?
- Which categories are most profitable?
- Is revenue concentrated in a small part of the assortment?
- Are high-revenue categories also high-margin?

During the initial data exploration, it became clear that product-level analysis in this dataset has several limitations.
Products with identical or very similar names are sometimes associated with different categories, and the catalog contains a very large number of individual SKUs with extremely low sales per item.

As a result, meaningful product-level insights are difficult to derive because sales are highly fragmented across many products.

For this reason, the analysis focuses primarily on **category-level performance**, which provides a clearer and more reliable view of revenue concentration and profitability patterns.

## 3.1 Data Extraction

The query below aggregates product and category-level sales, revenue, profit, and margin metrics.

In [16]:
query_product_analytics = """
SELECT
  p.name AS product_name,
  p.category,
  COUNT(oi.id) AS units_sold,
  SUM(oi.sale_price) AS revenue,
  SUM(oi.sale_price) - SUM(i.cost) AS profit,
  ROUND(SUM(oi.sale_price) / SUM(SUM(oi.sale_price)) OVER() * 100, 2) AS revenue_share_pct
FROM `bigquery-public-data.thelook_ecommerce.order_items` oi
JOIN `bigquery-public-data.thelook_ecommerce.products` p ON oi.product_id = p.id
JOIN `bigquery-public-data.thelook_ecommerce.inventory_items` i ON oi.inventory_item_id = i.id
WHERE DATE(oi.created_at) >= '2025-01-01'
  AND oi.status NOT IN ('Cancelled', 'Returned')
GROUP BY 1, 2
ORDER BY revenue DESC
"""

df_products = client.query(query_product_analytics).to_dataframe()

## 3.2 Top Categories by Revenue

This chart ranks categories by revenue and colors them by gross margin.

It highlights not only which categories are largest, but also whether scale is accompanied by profitability.

In [18]:
#@title Top Categories by Revenue

# Grouping data
df_categories = df_products.groupby('category').agg({
    'revenue': 'sum',
    'profit': 'sum'
}).reset_index()

# Calculate Margin %
df_categories['margin_pct'] = (df_categories['profit'] / df_categories['revenue'] * 100).round(2)

# Sorting by revenue (now for a horizontal chart it's better to have the leaders at the top)
df_categories = df_categories.sort_values(by='revenue', ascending=True)

# Plotting a horizontal chart (H-bar)
fig = px.bar(
    df_categories,
    x='revenue',
    y='category',
    color='margin_pct',    # The color reflects the margin
    orientation='h',
    text='revenue',
    title='Top Categories by Revenue (Color by Margin %)',
    labels={'revenue': 'Revenue ($)', 'margin_pct': 'Margin (%)', 'category': ''},
    color_continuous_scale='Viridis'
)

# Customizing text and appearance
fig.update_traces(
    texttemplate='$%{text:.3s}',
    textposition='outside',
    hovertemplate="<b>Category:</b> %{y}<br>" +
                  "<b>Revenue:</b> $%{x:,.0f}<br>" +
                  "<b>Margin:</b> %{customdata:.1f}%<extra></extra>",
    # Pass the margin data to customdata so it is available to the template
    customdata=df_categories['margin_pct']
)

fig.update_layout(
    margin=dict(l=150), # Increase the left indentation so that the category names fit
    template='plotly_white',
    height=800, # Increase the height so that the category list is not too dense
    coloraxis_colorbar=dict(title="Margin %"),
    barcornerradius=15
)

fig.show()

## 3.3 Product Portfolio Matrix

This matrix compares revenue and margin across categories.

It helps distinguish between:
- high-revenue / high-margin categories,
- high-revenue / low-margin categories,
- and smaller categories with stronger profitability.

In [20]:
#@title Product Portfolio Matrix

# Prepare the data
df_scatter = df_products.groupby('category').agg({
    'revenue': 'sum',
    'profit': 'sum',
    'units_sold': 'sum'
}).reset_index()

df_scatter['margin_pct'] = (df_scatter['profit'] / df_scatter['revenue'] * 100).round(2)

# Draw Bubble Chart
fig = px.scatter(
    df_scatter,
    x='revenue',
    y='margin_pct',
    size='units_sold',
    color='category',
    hover_name='category',
    text='category', # Add labels directly to the chart
    title='Product Portfolio Matrix: Revenue vs Margin',
    labels={
        'revenue': 'Total Revenue ($)',
        'margin_pct': 'Gross Margin (%)',
        'units_sold': 'Units Sold'
    },
    size_max=60 # Adjust the maximum bubble size
)

# Design settings
fig.update_traces(textposition='top center', # To make the text above the circle
)

# Add centerlines to divide the graph into 4 quadrants
avg_revenue = df_scatter['revenue'].mean()
avg_margin = df_scatter['margin_pct'].mean()

fig.add_hline(y=avg_margin, line_dash="dash", line_color="grey", annotation_text="Avg Margin")
fig.add_vline(x=avg_revenue, line_dash="dash", line_color="grey", annotation_text="Avg Revenue")

fig.update_layout(
    template='plotly_white',
    height=700,
    showlegend=False # Hide the legend since the names are already on the chart
)

fig.show()

## 3.4 Pareto Analysis: Revenue Concentration

The Pareto chart below shows how revenue is concentrated across categories.

The bars represent category revenue, while the cumulative line indicates what share of total revenue is explained by the top categories.

This is useful for identifying whether the business is broadly diversified or heavily dependent on a small number of categories.

In [24]:
#@title Pareto Analysis: Revenue Concentration

# Preparation (sorting is required for Pareto)
df_pareto = df_products.groupby('category').agg({'revenue': 'sum'}).reset_index()
df_pareto = df_pareto.sort_values(by='revenue', ascending=False)

# Accumulated amount
df_pareto['cum_revenue'] = df_pareto['revenue'].cumsum()
total_rev = df_pareto['revenue'].sum()
df_pareto['cum_pct'] = (df_pareto['cum_revenue'] / total_rev * 100)

# Creating Plot
fig = go.Figure()

# Add bar chart (Y1 axis)
fig.add_trace(go.Bar(
    x=df_pareto['category'],
    y=df_pareto['revenue'],
    name='Revenue',
    marker_color='royalblue',
    marker_pattern_shape="",
    marker_line_width=0,
    marker_cornerradius=10,    # Adding rounded corners
    yaxis='y1',
    hovertemplate="<b>Revenue:</b> $%{y:,.0f}<extra></extra>"
))

# Accumulation line (Y2 axis)
fig.add_trace(go.Scatter(
    x=df_pareto['category'],
    y=df_pareto['cum_pct'],
    name='Cumulative %',
    line=dict(color='firebrick', width=3),
    yaxis='y2',
    # Format the accumulated percentage with one decimal place
    hovertemplate="<b>Cumulative:</b> %{y:.1f}%<extra></extra>"
))

# Setup Layout
fig.update_layout(
    title='Pareto Analysis: Revenue Concentration',
    template='plotly_white',
    hovermode="x unified",
    hoverlabel=dict(bgcolor="rgba(255,255,255,0.9)", font_size=13),

    xaxis=dict(title='Product Category', tickangle=-45),

    yaxis=dict(
        title='Revenue ($)',
        side='left',
        showgrid=False
    ),

    yaxis2=dict(
        title='Cumulative Percentage (%)',
        side='right',
        overlaying='y',
        range=[0, 105],
        anchor='x',
        showgrid=True
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Add the 80% line
fig.add_shape(
    type="line",
    x0=0, x1=1, xref="paper",
    y0=80, y1=80, yref="y2",
    line=dict(color="green", width=2, dash="dot")
)

fig.show()

### Key Takeaways

- The top categories such as *Outerwear & Coats*, *Jeans*, and *Sweaters* generate a substantial share of total revenue.

- The Pareto analysis indicates that revenue is moderately concentrated, with roughly half of the categories contributing about 80% of total revenue.

- Some of the highest-revenue categories do not necessarily have the highest margins. For example, categories like *Jeans* generate significant revenue but operate with comparatively lower margins.

- The portfolio matrix highlights a few categories that combine both high revenue and strong margins, such as *Outerwear & Coats*, making them particularly valuable for the business.

- Smaller categories sometimes show higher margins but contribute relatively little to overall revenue, suggesting potential opportunities for growth if demand can be expanded.

# 4. Customer Behavior

This section focuses on customer journey events and conversion.

The aim is to understand:
- how users progress through the purchase funnel,
- how conversion changes over time,
- and how order outcomes are distributed.

## 4.1 Data Extraction

The following queries use event-level data to analyze funnel progression and purchase conversion over time.

In [26]:
query_order_status_analytics = """
  SELECT
  CASE WHEN status IN ('Complete', 'Processing', 'Shipped') THEN 'processed_orders'
  WHEN status = 'Cancelled' THEN 'cancelled_orders'
  WHEN status = 'Returned' THEN 'returned_orders'
  END AS status,
  COUNT(DISTINCT order_id) AS num_orders
  FROM `bigquery-public-data.thelook_ecommerce.orders`
  WHERE  DATE(created_at) >= '2025-01-01'
  GROUP BY 1
"""

df_status = client.query(query_order_status_analytics).to_dataframe()


query_funnel = """
WITH base_counts AS (
  SELECT
    COUNT(DISTINCT CASE WHEN event_type IN ('home', 'department', 'product', 'cart', 'purchase') THEN session_id END) as total_visitors,
    COUNT(DISTINCT CASE WHEN event_type IN ('product', 'cart', 'purchase') THEN session_id END) as product_viewers,
    COUNT(DISTINCT CASE WHEN event_type IN ('cart', 'purchase') THEN session_id END) as cart_adders,
    COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN session_id END) as purchasers
  FROM `bigquery-public-data.thelook_ecommerce.events`
  WHERE created_at >= '2025-01-01'
  AND event_type IN ('home', 'department', 'product', 'cart', 'purchase')
)
SELECT '1. All Visitors' as step, total_visitors as sessions FROM base_counts
UNION ALL
SELECT '2. Product Viewers' as step, product_viewers FROM base_counts
UNION ALL
SELECT '3. Add to Cart' as step, cart_adders FROM base_counts
UNION ALL
SELECT '4. Purchase' as step, purchasers FROM base_counts
"""
df_funnel = client.query(query_funnel).to_dataframe()



query_convertion_over_time = """
WITH sessions_by_day AS (
  SELECT
    DATE(created_at) AS event_date,
    COUNT(DISTINCT CASE
      WHEN event_type IN ('home', 'department', 'product', 'cart', 'purchase')
      THEN session_id
    END) AS total_sessions,
    COUNT(DISTINCT CASE
      WHEN event_type = 'purchase'
      THEN session_id
    END) AS purchasing_sessions
  FROM `bigquery-public-data.thelook_ecommerce.events`
  WHERE created_at >= '2025-01-01'
    AND event_type IN ('home', 'department', 'product', 'cart', 'purchase')
  GROUP BY 1
)

SELECT
  event_date,
  total_sessions,
  purchasing_sessions,
  SAFE_DIVIDE(purchasing_sessions, total_sessions) AS purchase_conversion_rate
FROM sessions_by_day
ORDER BY event_date
"""

df_cr_time = client.query(query_convertion_over_time).to_dataframe()

## 4.2 Order Status Distribution

This chart shows the share of processed, cancelled, and returned orders.

It provides a high-level operational view of order outcomes and helps put conversion in the context of order quality.

In [28]:
#@title Order Status Distribution


# Adjusting colors and offsets (pull)
# To make the donut "uneven," we pull out the Cancelled and Returned sectors
colors = ['#00CC96', '#EF553B', '#AB63FA']
pull_values = [0, 0.1, 0.15] # Offset: 0 for base, and increasing for problematic statuses
statuses_map = {
    'cancelled_orders' : 'Cancelled Orders',
    'processed_orders': 'Processed Orders',
    'returned_orders': 'Returned Orders'
}

df_status['display_status'] = df_status['status'].map(statuses_map)

fig = go.Figure(data=[go.Pie(
    labels=df_status['display_status'],
    values=df_status['num_orders'],
    hole=.6, # Hole size (makes the graph look like a donut)
    pull=pull_values,
    marker=dict(colors=colors, line=dict(color='#FFFFFF', width=2)),
    textinfo='percent+label',
    hovertemplate="<b>%{label}</b><br>Orders: %{value}<br>Share: %{percent}<extra></extra>",
    insidetextorientation='radial'
)])

# Setting up an appearance
fig.update_layout(
    title_text="Order Status Distribution 2025",
    annotations=[dict(text='Total Orders', x=0.5, y=0.5, font_size=20, showarrow=False)],
    showlegend=False,
    template='plotly_white',
    margin=dict(t=50, b=20, l=20, r=20),
)

fig.show()

## 4.3 Conversion Funnel

The funnel summarizes progression from visitor activity to purchase.

It highlights where the biggest drop-offs occur and provides a snapshot of overall conversion performance.

In [36]:
#@title Conversion Funnel

# Calculate the % from the first step (Initial Conversion)
first_step_value = df_funnel['sessions'].iloc[0]
df_funnel['pct_initial'] = (df_funnel['sessions'] / first_step_value * 100).round(1)

# Create a nice string to display inside the sectors
df_funnel['text_label'] = (
    df_funnel['step'] +
    "<br>" +
    df_funnel['sessions'].apply(lambda x: f"{x:,}") +
    " (" + df_funnel['pct_initial'].astype(str) + "%)"
)

fig = go.Figure(go.Funnelarea(
    values = df_funnel['sessions'],
    labels = df_funnel['step'],
    # Pass our prepared signatures to the text parameter
    text = df_funnel['text_label'],
    # Ask to display 'text', not the automatic 'percent'
    textinfo = "text",
    marker = {"colors": ["#003f5c", "#58508d", "#bc5090", "#ff6361", "#ffa600"]},
    showlegend = False
))

fig.update_layout(
    title_text="Conversion Funnel",
    template='plotly_white'
)

# Configure the pop-up window (hover) so that it also has the logic "from the first step"
fig.update_traces(
    customdata = df_funnel['pct_initial'],
    hovertemplate = "<b>Step:</b> %{label}<br>" +
                    "<b>Sessions:</b> %{value:,}<br>" +
                    "<b>Conversion:</b> %{customdata}% of initial<extra></extra>"
)

fig.show()

## 4.4 Purchase Conversion Rate Trend

This chart tracks purchase conversion rate over time using both daily values and a 7-day moving average.

The moving average helps smooth daily volatility and reveal the underlying trend more clearly.

In [38]:
#@title Purchase Conversion Rate Trend


# Data Preparation
df_cr_time['event_date'] = pd.to_datetime(df_cr_time['event_date'])
df_cr_time = df_cr_time.sort_values('event_date')

# Convert to percentages for convenience
df_cr_time['conversion_pct'] = df_cr_time['purchase_conversion_rate'] * 100

# Calculate the moving average for 7 days
df_cr_time['ma_7d'] = df_cr_time['conversion_pct'].rolling(window=7).mean()

# Create Plot
fig = go.Figure()

# Main line (thin and a little pale)
fig.add_trace(go.Scatter(
    x=df_cr_time['event_date'],
    y=df_cr_time['conversion_pct'],
    mode='lines',
    name='Daily Conversion',
    line=dict(color='rgba(31, 119, 180, 0.3)', width=1.5),
    hovertemplate="Daily Conversion: %{y:,.0f}%<extra></extra>"
))

# Moving average (bold accent line)
fig.add_trace(go.Scatter(
    x=df_cr_time['event_date'],
    y=df_cr_time['ma_7d'],
    mode='lines',
    name='7-Day Moving Avg',
    line=dict(color='#1f77b4', width=4),
    hovertemplate="7-Day Moving Avg: %{y:,.0f}%<extra></extra>"
))


fig.update_layout(
    title='Purchase Conversion Rate Trend (Daily vs 7-Day MA)',
    xaxis_title='Date',
    yaxis_title='Conversion Rate (%)',
    template='plotly_white',
    hovermode='x unified',
    yaxis=dict(ticksuffix="%", range=[0, df_cr_time['conversion_pct'].max() * 1.2]) # запас сверху
)

# Add a horizontal line of the average value for the entire period
avg_total = df_cr_time['conversion_pct'].mean()
fig.add_hline(y=avg_total, line_dash="dot", line_color="grey",
              annotation_text=f"Total Avg: {avg_total:.2f}%")

fig.show()

### Key Takeaways
- Funnel analysis shows where user drop-off is strongest.
- Order outcome distribution adds useful context beyond pure purchase conversion.
- The rolling conversion trend makes it easier to distinguish short-term volatility from actual performance changes.